# Notebook 24 — The Hub, Model Cards, and Inference

    ## Learning objectives

    - Inspect model metadata, revisions, licenses, and intended use
- Load models locally without leaking credentials
- Use Hugging Face Inference Providers through one client

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['transformers>=4.51,<5', 'huggingface-hub>=0.30,<1', 'sentencepiece']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 24.1 Artifacts and trust

A model repository can contain weights, configuration, tokenizer files, generation
defaults, chat templates, and custom code. Read the model card and license; pin a commit
revision for reproducibility. `trust_remote_code=True` executes repository code and
should be a deliberate security decision, not copied boilerplate.


In [ ]:
from huggingface_hub import HfApi, model_info
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
info = model_info(MODEL_ID)
print("model:", info.id)
print("sha:", info.sha)
print("pipeline:", info.pipeline_tag)
print("license:", (info.card_data or {}).get("license", "not declared"))
print("downloads:", info.downloads)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
device = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto").to(device)
messages = [{"role": "user", "content": "Explain a KV cache in one sentence."}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
new_tokens = output[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


## 24.2 Remote inference without provider-specific application code

`InferenceClient` routes requests to supported providers. Availability, structured
output, pricing, and limits vary by model/provider, so keep the model configurable and
handle errors explicitly. Tokens belong in `.env`, never notebooks or model prompts.


In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()
token = os.getenv("HUGGINGFACE_TOKEN") or os.getenv("HF_TOKEN")
chat_model = os.getenv("HF_CHAT_MODEL", "Qwen/Qwen2.5-7B-Instruct-1M")
hf = InferenceClient(token=token) if token else None
print("Remote client ready:", hf is not None, "model:", chat_model)
# Deliberately no remote call on import; run when your account/model supports it.


## 24.3 Repository anatomy and reproducibility

`config.json` describes architecture, but model-specific fields still require compatible
Transformers code. Tokenizer artifacts may include `tokenizer.json`, vocabulary/merges or a
SentencePiece model, special-token mappings, and `tokenizer_config.json` containing a chat
template. Generation defaults can live in `generation_config.json`. Weights are commonly
sharded Safetensors files plus an index. Processor files configure image preprocessing.

A model ID points to a mutable repository branch unless `revision` is pinned to a commit.
Pin model, tokenizer, dataset, and code revisions; store the resolved commit. Safetensors
avoids arbitrary pickle execution for weights, but `trust_remote_code=True` imports Python
from the repository. Review/pin that code and run it with appropriate isolation. Model cards
should state training data, intended use, limitations, license, metrics, and environmental or
ethical considerations, but completeness varies. Absence of a warning is not evidence of
suitability.


In [ ]:
# List files and classify the artifact surface without downloading weights.
from huggingface_hub import list_repo_files
files = list_repo_files(MODEL_ID, repo_type="model", revision=info.sha)
groups = {"weights": [], "tokenizer": [], "config": [], "code": [], "other": []}
for name in files:
    lower = name.lower()
    if lower.endswith((".safetensors", ".bin", ".gguf")): group = "weights"
    elif any(x in lower for x in ["tokenizer", "vocab", "merges", "sentencepiece"]): group = "tokenizer"
    elif lower.endswith((".json", ".yaml", ".yml")): group = "config"
    elif lower.endswith(".py"): group = "code"
    else: group = "other"
    groups[group].append(name)
for group, names in groups.items(): print(group, names[:8], "..." if len(names) > 8 else "")


## 24.4 Loading, devices, dtypes, and memory

`from_pretrained` resolves config, downloads/caches files, constructs modules, and loads
tensors. `dtype="auto"` follows checkpoint/config behavior; inspect actual parameter dtypes.
`.to(device)` places the complete model on one device. `device_map="auto"` uses Accelerate
to place layers across devices/CPU, which is useful for fitting but can be slow and is not a
training strategy. Quantization configurations alter storage and kernels. `low_cpu_mem_usage`
and sharded loading reduce temporary host memory.

`model.eval()` disables dropout but does not disable gradients; use `torch.inference_mode()`.
Input tensors must share a compatible device with the first model layers. Decode only newly
generated token IDs, not the full prompt, when presenting output. Inspect finish condition and
output length. Cache location and revision pinning matter in ephemeral Colab environments;
gated models require license acceptance and token permissions before download.


In [ ]:
# Inspect the loaded model rather than trusting requested settings.
first = next(model.parameters())
parameter_count = sum(p.numel() for p in model.parameters())
bytes_used = sum(p.numel() * p.element_size() for p in model.parameters())
print("class:", model.__class__.__name__)
print("parameters:", f"{parameter_count:,}")
print("parameter storage:", f"{bytes_used/2**20:.1f} MiB")
print("first parameter dtype/device:", first.dtype, first.device)
print("context configured:", getattr(model.config, "max_position_embeddings", None))
print("attention implementation:", getattr(model.config, "_attn_implementation", None))


## 24.5 Local, Inference Providers, Endpoints, and dedicated serving

Local Transformers is transparent and ideal for learning or batch experiments, but it does
not provide multi-user scheduling. Inference Providers route a common client call to hosted
providers; supported models/features and billing vary. Dedicated Hugging Face Inference
Endpoints provide managed replicas for chosen models. TGI and vLLM are specialized servers.
Choose based on model support, privacy boundary, hardware control, concurrency, latency/SLA,
observability, autoscaling, and sustained utilization—not “open versus closed” alone.

Build an application adapter that owns model ID, revision, timeout, retry policy, decoding,
schema validation, and normalized usage/error records. Remote failures include authentication,
gating, no compatible provider, cold starts, quota/rate limits, transient 5xx, unsupported
structured output/tools, and context overflow. Catch specific error classes where possible and
preserve a safe diagnostic. Never silently switch models in a quality-sensitive workflow.

**Model selection reference:** validate license; inspect tokenizer/template; establish memory;
run task evals; benchmark prompt/output distributions; test safety/languages; pin revision;
and record operational compatibility before promotion.


## 24.6 Hugging Face artifact/client reference

| Object | Purpose |
|---|---|
| `HfApi` | Repository/search/metadata operations |
| `model_info` | Card, tags, resolved SHA, provider metadata |
| `snapshot_download` | Materialize pinned repository snapshot |
| `AutoConfig` | Architecture configuration without full weights |
| `AutoTokenizer/Processor` | Text or multimodal preprocessing/template |
| `AutoModel*` | Task-specific model construction/loading |
| `InferenceClient` | Routed remote inference interface |

Cache reproducibility requires recording the resolved commit, not relying on whatever is currently in
the local cache. Use tokens through environment/secret stores and minimum permission. Accept gated
licenses deliberately. Review repository Python before remote-code execution. Model card benchmark
numbers may use different prompts, precision, templates, or contaminated data; rerun relevant evals.

On load failures check network/auth/gating, revision/file availability, disk, host RAM, device RAM,
dtype/quantization support, library architecture support, tokenizer/template, and custom-code version.
Distinguish download/load/generation time in benchmarks.


## 24.6 Resolve and record immutable revisions

A Hub branch or tag is mutable; a reproducible run records the resolved commit for model, tokenizer, processor, configuration, and datasets. Download only expected files, prefer Safetensors, inspect custom modeling code before enabling it, and record library versions. Offline reload is a powerful artifact test. Model cards and licenses inform suitability but do not replace organizational review. Gated access credentials belong in environment or Colab Secrets and should never enter notebook output, configuration, URLs, or uploaded repositories.


In [ ]:
manifest={"model_id":"org/model","requested_revision":"main","resolved_commit":"0123456789abcdef","trust_remote_code":False,"weight_format":"safetensors"}
required={"model_id","resolved_commit","trust_remote_code","weight_format"}; assert required<=manifest.keys(); print(manifest)


## 24.7 Inspect architecture before inference

Use configuration metadata to estimate parameter scale, attention type, context limits, special-token IDs, and compatibility before allocating weights. Confirm whether a repository is base, instruct, embedding, reranking, vision-language, dense, or MoE. Device maps distribute modules but do not guarantee a valid or fast placement; inspect them after loading. Compare parameter dtype with input and cache dtypes. Run a one-prompt smoke test through both the high-level pipeline and direct tokenizer/model path, then verify the rendered prompt and decoded continuation rather than accepting plausible text.


In [ ]:
config={"architectures":["CausalLM"],"hidden_size":2048,"num_hidden_layers":24,"num_attention_heads":16,"num_key_value_heads":4,"max_position_embeddings":32768}
assert config["hidden_size"]%config["num_attention_heads"]==0
print("head dim",config["hidden_size"]//config["num_attention_heads"],"GQA ratio",config["num_attention_heads"]//config["num_key_value_heads"])


## Exercises

    1. Pin a model revision and prove subsequent loads use it.
2. Compare model-card claims with a small task-specific evaluation.
3. Write a loader that refuses undeclared or disallowed licenses.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
